In [1]:
!pip install tiktoken


[notice] A new release of pip is available: 23.2.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [3]:
import os
import re
import json
import uuid
from pathlib import Path
from typing import List, Dict
import tiktoken

INPUT_FOLDER = "/workspaces/ai-research-platform/llm_lab/results"
OUTPUT_FOLDER = "/workspaces/ai-research-platform/llm_lab/chunks"

MAX_TOKENS = 500
MIN_TOKENS = 80

Path(OUTPUT_FOLDER).mkdir(exist_ok=True)

# Tokenizer (OpenAI-compatible)
tokenizer = tiktoken.get_encoding("cl100k_base")

def token_count(text: str) -> int:
    return len(tokenizer.encode(text))


# -------------------------------------------------
# Document Type Detection
# -------------------------------------------------
def detect_document_type(text: str) -> str:

    t = text.lower()

    if any(k in t for k in ["semester", "grade", "credits", "cgpa", "transcript", "examination"]):
        return "transcript"

    if any(k in t for k in ["skills", "experience", "projects", "education", "linkedin"]):
        return "cv"

    if any(k in t for k in ["certificate", "certified that", "provisional certificate", "controller of examinations"]):
        return "certificate"

    if any(k in t for k in ["invoice", "total amount", "billing", "payment due"]):
        return "invoice"

    if any(k in t for k in ["public services card", "passport", "id number", "date of expiry"]):
        return "id"

    return "generic"


# -------------------------------------------------
# Section Splitter
# -------------------------------------------------
def split_sections(text: str) -> List[str]:

    # Split on ALL CAPS headings or large headings
    pattern = r"\n(?=[A-Z][A-Z\s]{3,})"

    sections = re.split(pattern, text)

    return [s.strip() for s in sections if s.strip()]


# -------------------------------------------------
# Paragraph Splitter
# -------------------------------------------------
def split_paragraphs(section: str) -> List[str]:

    paragraphs = re.split(r"\n\s*\n", section)

    return [p.strip() for p in paragraphs if p.strip()]


# -------------------------------------------------
# Token Limit Chunking
# -------------------------------------------------
def enforce_token_limit(paragraphs: List[str]) -> List[str]:

    chunks = []
    current_chunk = []
    current_tokens = 0

    for para in paragraphs:

        tokens = token_count(para)

        if current_tokens + tokens <= MAX_TOKENS:

            current_chunk.append(para)
            current_tokens += tokens

        else:

            if current_chunk:
                chunks.append("\n\n".join(current_chunk))

            current_chunk = [para]
            current_tokens = tokens

    if current_chunk:
        chunks.append("\n\n".join(current_chunk))

    return chunks


# -------------------------------------------------
# Chunking Strategies
# -------------------------------------------------
def chunk_transcript(text: str):

    sections = split_sections(text)

    chunks = []

    for sec in sections:

        paragraphs = split_paragraphs(sec)

        chunks.extend(enforce_token_limit(paragraphs))

    return chunks


def chunk_cv(text: str):

    sections = split_sections(text)

    chunks = []

    for sec in sections:

        paragraphs = split_paragraphs(sec)

        chunks.extend(enforce_token_limit(paragraphs))

    return chunks


def chunk_certificate(text: str):

    # certificates are usually small
    if token_count(text) < MAX_TOKENS:
        return [text]

    paragraphs = split_paragraphs(text)

    return enforce_token_limit(paragraphs)


def chunk_generic(text: str):

    paragraphs = split_paragraphs(text)

    return enforce_token_limit(paragraphs)


# -------------------------------------------------
# Main Chunk Function
# -------------------------------------------------
def chunk_document(text: str, doc_name: str):

    doc_type = detect_document_type(text)

    if doc_type == "transcript":
        raw_chunks = chunk_transcript(text)

    elif doc_type == "cv":
        raw_chunks = chunk_cv(text)

    elif doc_type == "certificate":
        raw_chunks = chunk_certificate(text)

    else:
        raw_chunks = chunk_generic(text)

    final_chunks = []

    chunk_index = 0

    for c in raw_chunks:

        if token_count(c) < MIN_TOKENS:
            continue

        final_chunks.append(
            {
                "id": str(uuid.uuid4()),
                "document": doc_name,
                "document_type": doc_type,
                "chunk_index": chunk_index,
                "text": c
            }
        )

        chunk_index += 1

    return final_chunks


# -------------------------------------------------
# Process all documents
# -------------------------------------------------
def process_documents():

    files = list(Path(INPUT_FOLDER).glob("*.txt"))

    all_chunks = []

    for file in files:

        print(f"Processing {file.name}")

        text = Path(file).read_text()

        chunks = chunk_document(text, file.name)

        all_chunks.extend(chunks)

    output_file = Path(OUTPUT_FOLDER) / "chunks.json"

    with open(output_file, "w") as f:
        json.dump(all_chunks, f, indent=2)

    print(f"\nSaved {len(all_chunks)} chunks → {output_file}")


if __name__ == "__main__":

    process_documents()

Processing 18WJ1A05H4_btech.txt
Processing SCAN_20260306_172603684.txt
Processing SCAN_20260306_171008433.txt
Processing SCAN_20260306_172711773.txt
Processing SCAN_20260306_172642207.txt
Processing 18WJ1A05H4.txt
Processing Transcript (2).txt
Processing SRIKAR_KURAGAYALA_AI_cv.txt
Processing SCAN_20260306_172848415.txt
Processing SCAN_20260306_173032732.txt

Saved 27 chunks → /workspaces/ai-research-platform/llm_lab/chunks/chunks.json


In [4]:
!pip install sentence-transformers


[notice] A new release of pip is available: 23.2.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [5]:
import json
from pathlib import Path
import chromadb
from sentence_transformers import SentenceTransformer

CHUNKS_FILE = "/workspaces/ai-research-platform/llm_lab/chunks/chunks.json"
VECTOR_DB_PATH = "/workspaces/ai-research-platform/llm_lab/srikar_db"

# load embedding model
model = SentenceTransformer("all-MiniLM-L6-v2")

# create chroma client
client = chromadb.PersistentClient(path=VECTOR_DB_PATH)

collection = client.get_or_create_collection(
    name="documents"
)

# load chunks
with open(CHUNKS_FILE) as f:
    chunks = json.load(f)

print(f"Loaded {len(chunks)} chunks")


ids = []
documents = []
metadatas = []
embeddings = []

for chunk in chunks:

    text = chunk["text"]

    embedding = model.encode(text).tolist()

    ids.append(chunk["id"])

    documents.append(text)

    metadatas.append({
        "document": chunk["document"],
        "chunk_index": chunk["chunk_index"],
        "doc_type": chunk["document_type"]
    })

    embeddings.append(embedding)

print("Storing embeddings...")

collection.add(
    ids=ids,
    documents=documents,
    metadatas=metadatas,
    embeddings=embeddings
)

print("Done storing vectors.")

/workspaces/ai-research-platform/backend/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loaded 27 chunks
Storing embeddings...
Done storing vectors.


In [7]:
import chromadb
from sentence_transformers import SentenceTransformer

VECTOR_DB_PATH = "/workspaces/ai-research-platform/llm_lab/srikar_db"

model = SentenceTransformer("all-MiniLM-L6-v2")

client = chromadb.PersistentClient(path=VECTOR_DB_PATH)

collection = client.get_collection("documents")

query = "What is the CGPA of of srikar?"

embedding = model.encode(query).tolist()

results = collection.query(
    query_embeddings=[embedding],
    n_results=3
)

print(results["documents"])

[["* **SI.No.:** D205276\n* **Barcode:** 59/59292/D205276\n* **Name of Board:** Telangana State Board of Intermediate Education\n* **Address:** Vidya Bhavan, Nampally, Hyderabad - 500 001\n* **Student's Photo:** Included in the top right corner.\n* **Student Information:**\n\t+ **Name of Student:** Kuragayala Srikar\n\t+ **Father's Name:** Kuragayala Srihari\n\t+ **Mother's Name:** Kuragayala Kalpana\n\t+ **Registered No.:** 1859230347\n\t+ **Examination Details:** \n\t\t- **Exam:** Intermediate Public Examination\n\t\t- **Month & Year:** March-2018\n\t+ **Result:** \n\t\t- **Grade:** A Grade\n\t\t- **Medium of Instruction:** English\n* **Subjects and Marks:**\n\t+ **Part - I:**\n\t\t- **English:**\n\t\t\t- **I Year:** 94/100\n\t\t\t- **II Year:** 88/100\n\t+ **Part - II:**\n\t\t- **Sanskrit:**\n\t\t\t- **I Year:** 98/100\n\t\t\t- **II Year:** 99/100\n\t+ **Part - III (Optional Subjects):**\n\t\t- **Mathematics - A:**\n\t\t\t- **I Year:** 75/75\n\t\t\t- **II Year:** 75/75\n\t\t- **Math

In [8]:
!pip install scikit-learn


[notice] A new release of pip is available: 23.2.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [3]:
from pathlib import Path
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

DATA_FOLDER = "/workspaces/ai-research-platform/llm_lab/results"

documents = []
metadata = []

# -------------------------------------------------
# Load txt files and split into chunks
# -------------------------------------------------

for file in Path(DATA_FOLDER).glob("*.txt"):

    text = file.read_text()

    chunks = text.split("\n\n")

    for i, chunk in enumerate(chunks):

        if len(chunk.strip()) < 40:
            continue

        documents.append(chunk)

        metadata.append({
            "document": file.name,
            "chunk_index": i
        })


print("Loaded chunks:", len(documents))


# -------------------------------------------------
# Build TF-IDF search index
# -------------------------------------------------

vectorizer = TfidfVectorizer(stop_words="english")

doc_vectors = vectorizer.fit_transform(documents)

print("Search index built.")


# -------------------------------------------------
# Search function
# -------------------------------------------------

def search(query, top_k=10):

    query_vec = vectorizer.transform([query])

    scores = cosine_similarity(query_vec, doc_vectors)[0]

    top_indices = scores.argsort()[-top_k:][::-1]

    results = []

    for idx in top_indices:

        results.append({
            "score": float(scores[idx]),
            "document": metadata[idx]["document"],
            "chunk_index": metadata[idx]["chunk_index"],
            "text": documents[idx]
        })

    return results


# -------------------------------------------------
# Example query
# -------------------------------------------------

query = "how is srikars perfomence in acadamics?"

results = search(query)

for r in results:

    print("\n--------------------")
    print("Document:", r["document"])
    print("Chunk:", r["chunk_index"])
    print("Score:", r["score"])
    print(r["text"])

Loaded chunks: 167
Search index built.

--------------------
Document: SCAN_20260306_173032732.txt
Chunk: 9
Score: 0.0
* The document appears to be a formal academic certificate, specifically a master's degree certificate from Dublin City University.

--------------------
Document: SCAN_20260306_173032732.txt
Chunk: 7
Score: 0.0
* **Language:** The document features text in both English and Irish.

--------------------
Document: SCAN_20260306_173032732.txt
Chunk: 5
Score: 0.0
* **Signatures:**
	+ President: Dáire Keating
	+ Registrar: Una Joyce

--------------------
Document: SCAN_20260306_173032732.txt
Chunk: 3
Score: 0.0
* **Date of Conferring:** 1 October 2025
* **Irish Equivalent:** Dáta an Bhronnta

--------------------
Document: SCAN_20260306_173032732.txt
Chunk: 1
Score: 0.0
* **University Name:** Dublin City University
* **University Seal:** Features the text "OLLSCOIL CHATHAIR BHAILE ÁTHA CLIATH DUBLIN CITY UNIVERSITY" around a shield with "DCU"
* **Degree Information:**
	+ De

In [18]:
results = search("What is the student's degrees?")

print(results[0])

{'score': 0.40552283242333836, 'document': 'Transcript (2).txt', 'chunk_index': 45, 'text': '### 6.4 Doctoral Degrees are awarded without classification.'}


In [23]:
import json
from pathlib import Path
from groq import Groq
import chromadb
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import CrossEncoder
# -----------------------------
# Config
# -----------------------------

CHUNKS_FILE = "/workspaces/ai-research-platform/llm_lab/chunks/chunks.json"
VECTOR_DB_PATH = "./srikar_db"

client_llm = Groq()
reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")
# embedding model
embed_model = SentenceTransformer("all-MiniLM-L6-v2")

# -----------------------------
# Load chunks
# -----------------------------

with open(CHUNKS_FILE) as f:
    chunks = json.load(f)

documents = [c["text"] for c in chunks]
ids = [c["id"] for c in chunks]

print("Loaded chunks:", len(chunks))

# -----------------------------
# Build Vector DB
# -----------------------------

client = chromadb.PersistentClient(path=VECTOR_DB_PATH)

collection = client.get_or_create_collection("docs")

embeddings = embed_model.encode(documents).tolist()

collection.add(
    ids=ids,
    documents=documents,
    embeddings=embeddings,
)

print("Vector DB built")

# -----------------------------
# Build keyword search index
# -----------------------------

vectorizer = TfidfVectorizer(stop_words="english")
tfidf_matrix = vectorizer.fit_transform(documents)

print("Keyword index built")

# -----------------------------
# Query expansion
# -----------------------------

def expand_query(question):

    prompt = f"""
Generate 5 alternative search queries for the following question.

Question: {question}

Return only a list.
"""

    res = client_llm.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[{"role": "user", "content": prompt}],
    )

    text = res.choices[0].message.content

    queries = [q.strip("- ").strip() for q in text.split("\n") if q.strip()]

    return [question] + queries


# -----------------------------
# Vector retrieval
# -----------------------------

def vector_search(query, k=3):

    emb = embed_model.encode(query).tolist()

    res = collection.query(
        query_embeddings=[emb],
        n_results=k
    )

    return res["documents"][0]


# -----------------------------
# Keyword retrieval
# -----------------------------

def keyword_search(query, k=15):

    q_vec = vectorizer.transform([query])

    scores = cosine_similarity(q_vec, tfidf_matrix)[0]

    top_idx = scores.argsort()[-k:][::-1]

    return [documents[i] for i in top_idx]


# -----------------------------
# Hybrid retrieval
# -----------------------------

def retrieve(question):

    queries = expand_query(question)

    print("Expanded queries:", queries)

    results = []

    for q in queries:

        results += vector_search(q)
        results += keyword_search(q)

    # deduplicate
    unique = list(dict.fromkeys(results))

    return unique[:12]


# -----------------------------
# Build context
# -----------------------------

def build_context(chunks):

    context = ""

    for i, c in enumerate(chunks):

        context += f"\nDOCUMENT CHUNK {i+1}\n"
        context += c + "\n"

    return context

def rerank(query, chunks, top_k=5):

    pairs = [(query, chunk) for chunk in chunks]

    scores = reranker.predict(pairs)

    ranked = sorted(
        zip(chunks, scores),
        key=lambda x: x[1],
        reverse=True
    )

    top_chunks = [c[0] for c in ranked[:top_k]]

    return top_chunks

def retrieve_and_rerank(question):

    candidates = retrieve(question)  # from hybrid retrieval

    best_chunks = rerank(question, candidates)

    return best_chunks

def build_context(chunks):

    context = ""

    for i, chunk in enumerate(chunks):
        context += f"\nDOCUMENT CHUNK {i+1}\n"
        context += chunk + "\n"

    return context

# -----------------------------
# Answer question
# -----------------------------

def ask(question):

    chunks = retrieve_and_rerank(question)

    context = build_context(chunks)

    prompt = f"""
You are a document assistant.

Answer the question using ONLY the provided context this content is from my transcripts,cv and other documents like ids.

CONTEXT:
{context}

QUESTION:
{question}
"""

    response = client_llm.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[{"role":"user","content":prompt}]
    )

    return response.choices[0].message.content




Loaded chunks: 27
Vector DB built
Keyword index built


In [25]:
question = "is srikargood in acadamics?"

answer = ask(question)

print("\nANSWER:\n")
print(answer)

Expanded queries: ['is srikargood in acadamics?', '1. srikargood academic credentials', '2. srikargood education background check', '3. is srikargood a qualified academic', '4. srikargood academic achievements', '5. srikargood qualifications reviewed online']

ANSWER:

Based on the provided context, it can be inferred that Srikar is good in academics. 

His academic achievements include:

- A Grade in Intermediate Public Examination (March 2018) with a grand total of 970/1000 marks.
- First Class Honours in Bachelor of Technology: Computer Science (GPA: 8.3/10).
- First Class Honours in Master of Science in Computing: Artificial Intelligence (GPA: 1:1).

These achievements indicate that Srikar has consistently performed well in his academic pursuits.
